# 🤖 Class 6: Neo Opens SAP
### What is an AI agent, and how does it think? · Day 3 · Afternoon · 2 hours

**Same story.** This morning Neo learned to answer from the policy binder. Now a manager asks:
*"Why is PO 4500012345 delayed, and who should I contact?"*, and **the answer isn't in any document**. Neo has to check SAP, look up the vendor and apply the policy.

**Today's question:** *A chatbot answers. An agent acts. What's the difference, and how does an agent decide what to do?*

| Part | Topic | Cells |
|---|---|---|
| A | Chatbot vs agent | S, 1 |
| B | The 4 parts of an agent (and its tools) | 2–3 |
| C | How tools really work | 4 |
| D | The loop: think → act → observe | 5–7 |
| E | Your turn | 8–11 |

> 🔒 Mock SAP data only. The "SAP" here is two CSV files. Neo can use **only the tools we give it**.

## ⚙️ ▶ CELL S: Setup

In [1]:
# ▶ CELL S · SETUP: check Ollama. Run once.
import sys, os, re, json, csv
sys.path.insert(0, "../../shared")            # where bootcamp_llm.py lives
from bootcamp_llm import *

DATA_DIR = "../data"                          # orders.csv and vendors.csv (mock SAP data)
check_setup()
print("🤖 Neo says:", ask_ai("Reply with exactly these words: Connection OK", temperature=0, max_tokens=10))

✅ Ollama 0.40.3 is running at http://localhost:11434
✅ Main model : qwen3:8b
✅ Small model: llama3.2:1b
⚙️  Config: c:\Users\rajuk\OneDrive\Desktop\Day_3\bootcamp_config.json
🤖 Neo says: Connection OK


---
# A · Chatbot vs agent
**In real life:** to answer the manager properly you would (1) check the PO status, (2) find the block reason, (3) look up the vendor, (4) check the policy, (5) write the answer. **Five steps, different information at each step.**

### ▶ CELL 1: A chatbot with no tools

In [2]:
# ▶ CELL 1 · A chatbot: it only has what you give it
QUESTION = "Why is PO 4500012345 delayed, and who should I contact?"   # ← change me
show(ask_ai(QUESTION, temperature=0))

The tracking number **PO 4500012345** appears to be a **purchase order (PO)** number, not a tracking number for a shipment. Purchase orders are used internally by businesses to authorize purchases, and they are not typically used for tracking shipments or packages.

### Possible Reasons for Delay:
1. **Internal Processing**: The PO may be in the process of being fulfilled by the vendor or supplier.
2. **Vendor Delays**: The vendor may be experiencing delays in processing or shipping the order.
3. **Inventory Issues**: The item may be out of stock or delayed due to supply chain issues.
4. **Payment Issues**: The payment for the PO may not have been processed yet, causing a delay in fulfillment.

### Who to Contact:
- **Your Internal Purchasing Department**: If you're the one who placed the PO, contact the person or department responsible for managing purchase orders.
- **Vendor/Supplier**: Reach out to the vendor directly using the contact information provided in the PO or on their website.
- **Account Manager**: If you have an account manager with the vendor, they may be able to provide more details on the status of the PO.

### Next Steps:
1. Check the PO details to see if there is a contact person or department listed.
2. Contact the vendor using the contact information provided in the PO.
3. If you're unsure who to contact, reach out to your company’s procurement or purchasing team for assistance.

Let me know if you need help finding contact details for a specific vendor or PO.

> 🔎 **What to look for:** it cannot know the status or the vendor, so it either says so, or **invents** something plausible. A chatbot answers from what it was given. For this question we need something that can *fetch* information.

---
# B · The 4 parts of an agent
**Brain** (the LLM) + **Tools** (actions it can use) + **Memory** (steps so far) + **Instructions** (goal and rules).

### ▶ CELL 2: The tools: ordinary Python functions
These are Neo's "BAPIs". They are **not AI**: they read our mock data and always give the same answer.

In [3]:
# ▶ CELL 2 · TOOLS: ordinary functions that read our mock SAP data (no AI inside)
ORDERS  = {r["po_number"]: r for r in csv.DictReader(open(f"{DATA_DIR}/orders.csv", encoding="utf-8"))}
VENDORS = {r["vendor_id"]: r for r in csv.DictReader(open(f"{DATA_DIR}/vendors.csv", encoding="utf-8"))}

def check_order_status(po_number):
    r = ORDERS.get(str(po_number).strip())
    if not r:
        return {"error": f"PO {po_number} not found"}
    return {"po_number": r["po_number"], "status": r["status"], "block_reason": r["block_reason"] or None,
            "value_inr": int(r["value_inr"]), "delivery_date": r["delivery_date"], "vendor_id": r["vendor_id"]}

def get_vendor_details(po_number):
    r = ORDERS.get(str(po_number).strip())
    if not r:
        return {"error": f"PO {po_number} not found"}
    v = VENDORS[r["vendor_id"]]
    return {"vendor": v["vendor_name"], "contact": v["contact"], "email": v["email"], "rating": v["rating"]}

def check_policy(value_inr):
    value = int(re.sub(r"[^0-9]", "", str(value_inr)) or 0)       # accepts 85000, "85,000" or "₹85,000"
    if value <= 100000:   return {"value_inr": value, "approver": "Line Manager",    "policy": "3.1"}
    if value <= 1000000:  return {"value_inr": value, "approver": "Department Head", "policy": "3.2"}
    return                       {"value_inr": value, "approver": "Director",        "policy": "3.3"}

print(check_order_status("4500012345"))
print(get_vendor_details("4500012345"))
print(check_policy(85000), "\n", check_policy(1200000))

{'po_number': '4500012345', 'status': 'Blocked', 'block_reason': 'Credit check', 'value_inr': 50000, 'delivery_date': '2026-10-01', 'vendor_id': 'V100'}
{'vendor': 'Alpha Supplies', 'contact': 'Priya Sharma', 'email': 'priya@alpha.example', 'rating': 'A'}
{'value_inr': 85000, 'approver': 'Line Manager', 'policy': '3.1'} 
 {'value_inr': 1200000, 'approver': 'Director', 'policy': '3.3'}


> 🛠️ Same input, same output, every time. **The AI will not do the maths or the lookup. It will ask for a tool to do it.**

### ▶ CELL 3: The tool menu and the instructions
The agent knows its tools only from a short **menu** inside its instructions. Clear names and descriptions matter, just like a good prompt.

In [4]:
# ▶ CELL 3 · TOOL MENU + INSTRUCTIONS: the only things the agent knows about its tools
TOOLS = {
    "check_order_status": {"fn": check_order_status, "args": "po_number",
                           "does": "Status, block reason, value (INR), delivery date and vendor id of a purchase order."},
    "get_vendor_details": {"fn": get_vendor_details, "args": "po_number",
                           "does": "Vendor name, contact person, email and rating for a purchase order."},
    "check_policy":       {"fn": check_policy,       "args": "value_inr",
                           "does": "Who must approve a purchase order of this value (Procurement Policy section 3)."},
}

def system_prompt():
    menu = "\n".join(f"- {name}({t['args']}): {t['does']}" for name, t in TOOLS.items())
    return f"""You are Neo, an SAP procurement assistant for Nova Industries.
GOAL: answer the user's question using ONLY facts returned by your tools. Never guess a status, a contact or an approver.

TOOLS (the only ones you may use):
{menu}

Reply with ONE JSON object and nothing else.
To use a tool:  {{"thought": "why I need it", "tool": "<tool name>", "args": {{"<argument>": "<value>"}}}}
To finish:      {{"thought": "why I can finish", "answer": "<final answer for the user>"}}"""

print(system_prompt())

You are Neo, an SAP procurement assistant for Nova Industries.
GOAL: answer the user's question using ONLY facts returned by your tools. Never guess a status, a contact or an approver.

TOOLS (the only ones you may use):
- check_order_status(po_number): Status, block reason, value (INR), delivery date and vendor id of a purchase order.
- get_vendor_details(po_number): Vendor name, contact person, email and rating for a purchase order.
- check_policy(value_inr): Who must approve a purchase order of this value (Procurement Policy section 3).

Reply with ONE JSON object and nothing else.
To use a tool:  {"thought": "why I need it", "tool": "<tool name>", "args": {"<argument>": "<value>"}}
To finish:      {"thought": "why I can finish", "answer": "<final answer for the user>"}


---
# C · How tools really work
**The AI itself runs nothing.** It can only write text. So it writes a *request*, our program runs the tool, and hands back the result.

```
1. AI writes a REQUEST     {"tool": "check_order_status", "args": {"po_number": "4500012345"}}
2. The PROGRAM runs the real function
3. The PROGRAM gives the RESULT back to the AI
4. The AI carries on thinking
```
### ▶ CELL 4: One request, by hand

In [5]:
# ▶ CELL 4 · ONE tool request: the AI asks, the PROGRAM runs it
messages = [{"role": "system", "content": system_prompt()},
            {"role": "user", "content": QUESTION}]
request = chat_json(messages)
print("1️⃣ The AI wrote a REQUEST :", request)

if "tool" in request:
    result = TOOLS[request["tool"]]["fn"](**request["args"])
    print("2️⃣ The PROGRAM ran it     :", result)
    print("3️⃣ The result goes back to the AI, which then decides the next step.")
else:
    print("The model answered without using a tool. Run the cell again to see a tool request.")

1️⃣ The AI wrote a REQUEST : {'thought': 'I need to check the status of the purchase order to determine why it is delayed and then retrieve the vendor details to find the contact person.', 'tool': 'check_order_status', 'args': {'po_number': '4500012345'}}
2️⃣ The PROGRAM ran it     : {'po_number': '4500012345', 'status': 'Blocked', 'block_reason': 'Credit check', 'value_inr': 50000, 'delivery_date': '2026-10-01', 'vendor_id': 'V100'}
3️⃣ The result goes back to the AI, which then decides the next step.


> 🔑 **This is the control point.** The AI can only request tools on the menu. Unknown tool? Our program refuses (see Cell 7).

---
# D · The loop: Think → Act → Observe → Repeat
### ▶ CELL 5: The full agent
It fits on one screen. Each pass: the AI **thinks** and asks for a tool, the program **acts**, the AI **observes** the result, then repeats until it can answer.

In [6]:
# ▶ CELL 5 · THE AGENT LOOP: think → act → observe → repeat
def execute(reply):
    """The PROGRAM's side: run the requested tool, but only if it is on the allowed list."""
    name, args = reply.get("tool"), reply.get("args") or {}
    if name not in TOOLS:
        return {"error": f"Unknown tool '{name}'. Allowed tools: {list(TOOLS)}"}
    try:
        return TOOLS[name]["fn"](**args)
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Bad arguments for {name}: {e}"}

def run_agent(question, max_steps=5, verbose=True):
    messages = [{"role": "system", "content": system_prompt()}, {"role": "user", "content": question}]
    used = []
    for step in range(1, max_steps + 1):
        try:
            reply = chat_json(messages)
        except ValueError as e:                                   # the model broke the JSON format
            if verbose: print(f"⚠️  step {step}: {e}")
            messages.append({"role": "user", "content": "Reply with ONE valid JSON object only."})
            continue
        if not isinstance(reply, dict):                           # e.g. the model sent a list
            messages.append({"role": "user", "content": "Reply with ONE JSON object only."})
            continue
        if verbose: print(f"🧠 THINK   (step {step}): {reply.get('thought', '')}")
        if "answer" in reply:
            if verbose: print(f"✅ ANSWER : {reply['answer']}")
            return {"answer": str(reply["answer"]), "tools": used, "steps": step, "stopped": False}
        observation = execute(reply)                              # ACT
        used.append(reply.get("tool"))
        if verbose:
            print(f"🛠️  ACT     : {reply.get('tool')}({reply.get('args')})")
            print(f"👀 OBSERVE: {observation}")
        messages += [{"role": "assistant", "content": json.dumps(reply, ensure_ascii=False)},
                     {"role": "user", "content": "Tool result: " + json.dumps(observation, ensure_ascii=False)}]
    if verbose: print("⛔ Stopped: step limit reached")            # a guardrail (more in Class 7)
    return {"answer": None, "tools": used, "steps": max_steps, "stopped": True}

_ = run_agent(QUESTION)

🧠 THINK   (step 1): I need to check the status of the purchase order to determine why it is delayed and then retrieve the vendor details to find the contact person.
🛠️  ACT     : check_order_status({'po_number': '4500012345'})
👀 OBSERVE: {'po_number': '4500012345', 'status': 'Blocked', 'block_reason': 'Credit check', 'value_inr': 50000, 'delivery_date': '2026-10-01', 'vendor_id': 'V100'}
🧠 THINK   (step 2): I need to get the vendor details to find the contact person and check the procurement policy to determine who should approve the purchase order.
🛠️  ACT     : get_vendor_details({'po_number': '4500012345'})
👀 OBSERVE: {'vendor': 'Alpha Supplies', 'contact': 'Priya Sharma', 'email': 'priya@alpha.example', 'rating': 'A'}
🧠 THINK   (step 3): I need to check the procurement policy to determine who should approve the purchase order based on its value.
🛠️  ACT     : check_policy({'value_inr': 50000})
👀 OBSERVE: {'value_inr': 50000, 'approver': 'Line Manager', 'policy': '3.1'}
🧠 THINK   (s

> 🔁 Read it aloud as **THINK → ACT → OBSERVE**. Two or three passes, then the answer. *Nobody told it the order of the tools; it decided.* The wording of the THINK lines will differ on your model.

### ▶ CELL 6: Different questions, different tools

In [7]:
# ▶ CELL 6 · Three questions: does the agent pick different tools?
EXERCISES = [  # (question, tools we expect, a word the answer should contain)
    ("What is the status of PO 4500012345?",                                  {"check_order_status"},                     "block"),
    ("Who is the vendor for PO 4500012346 and how do I contact them?",        {"get_vendor_details"},                     "rahul"),
    ("Can PO 4500012347 be approved by a line manager?",                      {"check_order_status", "check_policy"},     "yes"),
]
for question, expected_tools, keyword in EXERCISES:
    r = run_agent(question, verbose=False)
    ok_tools = expected_tools <= set(r["tools"])
    ok_answer = bool(r["answer"]) and keyword in r["answer"].lower()
    print(f"{'✅' if ok_tools and ok_answer else '⚠️'} {question}")
    print(f"     tools used: {r['tools']}   steps: {r['steps']}   stopped: {r['stopped']}")
    print(f"     answer    : {(r['answer'] or '—')[:140]}\n")

✅ What is the status of PO 4500012345?
     tools used: ['check_order_status']   steps: 2   stopped: False
     answer    : The status of PO 4500012345 is 'Blocked' because of a credit check.

✅ Who is the vendor for PO 4500012346 and how do I contact them?
     tools used: ['get_vendor_details']   steps: 2   stopped: False
     answer    : The vendor for PO 4500012346 is Bharat Logistics. You can contact Rahul Verma at rahul@bharat.example.

✅ Can PO 4500012347 be approved by a line manager?
     tools used: ['check_order_status', 'check_policy']   steps: 3   stopped: False
     answer    : Yes, PO 4500012347 can be approved by a line manager as per the procurement policy for purchase orders valued at INR 85,000.



> ⚠️ A ⚠️ is not a failure of the class. It is the lesson: **agents can pick the wrong tool or misread a result**, so we test them.

### ▶ CELL 7: Safety on the menu: unknown tools, bad arguments, step limit

In [8]:
# ▶ CELL 7 · The control point: only tools on the menu run, and the loop has a step limit
print("1) A tool that is NOT on the menu:")
print("  ", execute({"tool": "delete_purchase_order", "args": {"po_number": "4500012345"}}))
print("2) Wrong arguments:")
print("  ", execute({"tool": "check_order_status", "args": {"order": "4500012345"}}))
print("3) A PO that does not exist:")
print("  ", execute({"tool": "check_order_status", "args": {"po_number": "4500099999"}}))

print("\n4) A step limit of 1 for a question that needs two or more steps:")
_ = run_agent("Why is PO 4500012345 delayed, and who should I contact?", max_steps=1)

1) A tool that is NOT on the menu:
   {'error': "Unknown tool 'delete_purchase_order'. Allowed tools: ['check_order_status', 'get_vendor_details', 'check_policy']"}
2) Wrong arguments:
   {'error': "Bad arguments for check_order_status: check_order_status() got an unexpected keyword argument 'order'"}
3) A PO that does not exist:
   {'error': 'PO 4500099999 not found'}

4) A step limit of 1 for a question that needs two or more steps:
🧠 THINK   (step 1): I need to check the status of the purchase order to determine why it is delayed and then retrieve the vendor details to find the contact person.
🛠️  ACT     : check_order_status({'po_number': '4500012345'})
👀 OBSERVE: {'po_number': '4500012345', 'status': 'Blocked', 'block_reason': 'Credit check', 'value_inr': 50000, 'delivery_date': '2026-10-01', 'vendor_id': 'V100'}
⛔ Stopped: step limit reached


> 🛡️ The program, not the AI, decides what runs. **The AI decides; the system executes, and only the tools we allow.** That is the idea we expand tomorrow (MCP and guardrails).

---
# E · Your turn (40 min)
Change only the lines marked **← change me**.

### ▶ CELL 8: Ask your own question

In [9]:
# ▶ CELL 8 · Your turn: ask the agent your own question
MY_QUESTION = "Who must approve PO 4500012348, and why is it blocked?"      # ← change me
_ = run_agent(MY_QUESTION)

🧠 THINK   (step 1): I need to check the approval policy for the value of the PO and the status of the PO to determine who must approve it and why it is blocked.
🛠️  ACT     : check_order_status({'po_number': '4500012348'})
👀 OBSERVE: {'po_number': '4500012348', 'status': 'Blocked', 'block_reason': 'Price variance', 'value_inr': 1200000, 'delivery_date': '2026-09-30', 'vendor_id': 'V300'}
🧠 THINK   (step 2): I need to determine the approver based on the value of the PO and confirm the block reason.
🛠️  ACT     : check_policy({'value_inr': 1200000})
👀 OBSERVE: {'value_inr': 1200000, 'approver': 'Director', 'policy': '3.3'}
🧠 THINK   (step 3): I have the necessary information to answer the user's question.
✅ ANSWER : The PO 4500012348 must be approved by the Director as per procurement policy 3.3. The PO is blocked due to a price variance.


> 💡 Try: *Who is the vendor for PO 4500012350?* · *What is the delivery date of PO 4500012354?* · *Is PO 4500012355 blocked, and who is the contact?*

### ▶ CELL 9: What if we remove a tool?
Take away `check_policy` and ask an approval question.

In [10]:
# ▶ CELL 9 · Your turn: remove a tool. What does the agent do without check_policy?
removed = TOOLS.pop("check_policy")                       # the tool leaves the menu
try:
    _ = run_agent("Who must approve PO 4500012348?", max_steps=4)
finally:
    TOOLS["check_policy"] = removed                       # put it back for the next cells
print("\n(check_policy is back on the menu:", "check_policy" in TOOLS, ")")

🧠 THINK   (step 1): I need to check the vendor details to find the contact person who might be involved in the approval process.
🛠️  ACT     : get_vendor_details({'po_number': '4500012348'})
👀 OBSERVE: {'vendor': 'Chennai Components', 'contact': 'Lakshmi Iyer', 'email': 'lakshmi@chennai.example', 'rating': 'B'}
🧠 THINK   (step 2): The contact person for the vendor is Lakshmi Iyer, who might be the one to approve the purchase order.
✅ ANSWER : Lakshmi Iyer

(check_policy is back on the menu: True )


> 🧪 Without the policy tool the agent either says it **cannot tell**, or **guesses**. (Correct answer: *Director*, ₹12,00,000.) *Tools are what make an agent reliable.*

### ▶ CELL 10: Add a new tool
A fourth tool that summarises delayed orders per vendor. We add it to the menu, and the agent can use it straight away.

In [11]:
# ▶ CELL 10 · Your turn: add a 4th tool to the menu
def delayed_summary():
    """Number and total value of Delayed purchase orders, per vendor."""
    out = {}
    for r in ORDERS.values():
        if r["status"] == "Delayed":
            row = out.setdefault(r["vendor_name"], {"delayed_pos": 0, "total_value_inr": 0})
            row["delayed_pos"] += 1
            row["total_value_inr"] += int(r["value_inr"])
    return out

TOOLS["delayed_summary"] = {"fn": delayed_summary, "args": "(none)",
                            "does": "Number and total value (INR) of Delayed purchase orders for every vendor."}
print("Tool check →", delayed_summary())

MY_QUESTION_2 = "Which vendor has the most delayed order value, and how much is it?"   # ← change me
_ = run_agent(MY_QUESTION_2)

Tool check → {'Bharat Logistics': {'delayed_pos': 2, 'total_value_inr': 1200000}, 'Chennai Components': {'delayed_pos': 2, 'total_value_inr': 1500000}}
🧠 THINK   (step 1): I need to use the delayed_summary tool to get the number and total value of delayed purchase orders for every vendor.
🛠️  ACT     : delayed_summary({})
👀 OBSERVE: {'Bharat Logistics': {'delayed_pos': 2, 'total_value_inr': 1200000}, 'Chennai Components': {'delayed_pos': 2, 'total_value_inr': 1500000}}
🧠 THINK   (step 2): I have the total value of delayed orders for each vendor. Bharat Logistics has a total value of 1,200,000 INR and Chennai Components has a total value of 1,500,000 INR. Chennai Components has the most delayed order value.
✅ ANSWER : Chennai Components has the most delayed order value, which is 1,500,000 INR.


> ✅ Expected: **Chennai Components, ₹15,00,000** (Bharat Logistics has ₹12,00,000). The tool does the adding; the agent only chooses to use it.

### ▶ CELL 11: Scoreboard: check every exercise in code

In [12]:
# ▶ CELL 11 · Final check: run all four exercise questions and score the tools used
FINAL = [  # (question, tools that must be used, word the answer must contain)
    ("What is the status of PO 4500012345?",                              {"check_order_status"},                 "block"),
    ("Who is the vendor for PO 4500012346 and how do I contact them?",    {"get_vendor_details"},                 "rahul"),
    ("Can PO 4500012347 be approved by a line manager?",                  {"check_order_status", "check_policy"}, "yes"),
    ("Who must approve PO 4500012348?",                                   {"check_policy"},                       "director"),
]
points = 0
for question, need, keyword in FINAL:
    r = run_agent(question, verbose=False)
    ok = need <= set(r["tools"]) and bool(r["answer"]) and keyword in r["answer"].lower()
    points += ok
    print(f"{'✅' if ok else '❌'} {question}   tools: {r['tools']}")
print(f"\nScore: {points} / {len(FINAL)}")

✅ What is the status of PO 4500012345?   tools: ['check_order_status']
✅ Who is the vendor for PO 4500012346 and how do I contact them?   tools: ['get_vendor_details']
✅ Can PO 4500012347 be approved by a line manager?   tools: ['check_order_status', 'check_policy']
✅ Who must approve PO 4500012348?   tools: ['check_order_status', 'check_policy']

Score: 4 / 4


---
## ✅ Remember three things
1. **Agent = Brain + Tools + Memory + Instructions.**
2. It works in a **loop: Think → Act → Observe → Repeat**.
3. **The AI decides; the system executes**, and only the tools we allow.

> 🌉 **Bridge to Day 4:** Our tools were built by hand for this one agent. A real company has dozens of systems. *How do agents connect to real systems in a standard way, and how do we keep them safe?*

---
## 🛠️ Trainer troubleshooting (Ollama)
| Message | Cause | Fix |
|---|---|---|
| `Cannot reach Ollama at http://localhost:11434` | Ollama isn't running | Windows: Start menu → **Ollama**. Mac: open the Ollama app. Or in a terminal: `ollama serve` |
| `Model '…' is not downloaded` | The model in `bootcamp_config.json` isn't pulled | `ollama pull <name>` (see `ollama list`) |
| `ModuleNotFoundError: bootcamp_llm` | The notebook is not running from its own folder | Open the **Day_3** folder as the workspace, then pick the **`.venv`** kernel |
| Very slow answers | The model is too big for this laptop | Choose a smaller `model` in `bootcamp_config.json`, then **Restart kernel** |
| `The model did not return valid JSON` (Class 6) | Small models sometimes break the format | Run the cell again. This is a real lesson: agents need validation |
| `embed` fails or all scores look the same (Class 5) | The embedding model is missing | `ollama pull nomic-embed-text`, or set `embed_model` to your chat model in the config |